In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)

n = 5000

monto_deuda = np.random.lognormal(mean=15, sigma=1.2, size=n)
antiguedad_meses = np.random.randint(1, 240, size=n)
tipo_impuesto = np.random.choice(["Predial", "Vehículos", "ICA", "Otros"], size=n, p=[0.35, 0.30, 0.20, 0.15])
tipo_deudor = np.random.choice(["Natural", "Jurídico"], size=n, p=[0.83, 0.17])

df = pd.DataFrame({
    "monto_deuda": monto_deuda,
    "antiguedad_meses": antiguedad_meses,
    "tipo_impuesto": tipo_impuesto,
    "tipo_deudor": tipo_deudor,
})

log_monto = np.log(df["monto_deuda"])
log_monto_normalizado = (log_monto - log_monto.min()) / (log_monto.max() - log_monto.min())
prob_pago = 0.75 - 0.5 * log_monto_normalizado
prob_pago = prob_pago.clip(0.05, 0.95)
df["sostuvo_pago"] = np.random.binomial(1, prob_pago)

X = df[["monto_deuda", "antiguedad_meses", "tipo_impuesto", "tipo_deudor"]]
y = df["sostuvo_pago"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [3]:
# dummies
X_train_encoded = pd.get_dummies(X_train, columns=["tipo_impuesto", "tipo_deudor"], drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=["tipo_impuesto", "tipo_deudor"], drop_first=True)

X_train_encoded.head()

,monto_deuda,antiguedad_meses,tipo_impuesto_Otros,tipo_impuesto_Predial,tipo_impuesto_Vehículos,tipo_deudor_Natural
1744,1.074296e+06,70,0,0,0,0
3541,1.864711e+07,239,0,1,0,1
1180,4.056670e+06,51,0,1,0,1
4992,2.848313e+06,141,0,0,1,1
4338,2.748323e+05,1,0,0,1,1


In [4]:
#ENTRENAR RANDOM FOREST DESPUES DE TENER LAS CATEGORICAS
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42
)

modelo.fit(X_train_encoded, y_train)

y_pred = modelo.predict(X_test_encoded)
y_proba = modelo.predict_proba(X_test_encoded)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precisión:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.542
Precisión: 0.5526315789473685
Recall: 0.7471910112359551
ROC-AUC: 0.5548817733198309
[[143 323]
 [135 399]]
